## 1. Method Choice and Why?

I selected Random Forest Classifier (or HistGradientBoostingClassifier) for Lane 1 (Content Opportunity Scoring):

WHY?
It handles non_linear relationships between search metrics(impressions, CTR, position) without needing feature scalling. It inherently prevents severe overfitting compared to a single deep Decision Tree. It allows computing Feature Importances to interpret which signals drive content decay predictions.

## 2. Split design

My cross-validation strategy is that I am using GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42) grouped on client_hash_id. The reason for this is that it prevents data contamination/leakage by ensuring all records belonging to a single client domain stay strictly in either the training set or the test set.

## 3. Train + Compare vs my baseline

In [ ]:
!pip install matplotlib
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# 1. Dynamically find the parquet file location
possible_paths = [
    "fact_content_daily_performance_sample.parquet",          # Inside work/notebooks/
    "../fact_content_daily_performance_sample.parquet",       # Inside work/
    "../../fact_content_daily_performance_sample.parquet",    # In repo root directory
    "work/fact_content_daily_performance_sample.parquet"
]

LOCAL_FILE = None
for path in possible_paths:
    if os.path.exists(path):
        LOCAL_FILE = path
        break

if LOCAL_FILE is None:
    raise FileNotFoundError(
        f"Could not find 'fact_content_daily_performance_sample.parquet'. "
        f"Current working directory is: {os.getcwd()}"
    )

print(f"File located successfully at: {LOCAL_FILE}")

# 2. Load Local Parquet Data via DuckDB
con = duckdb.connect()

query = f"""
SELECT 
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS avg_ctr,
    AVG(gsc_avg_position) AS avg_position,
    COUNT(DISTINCT report_date) / 30.0 AS active_days_ratio,
    
    -- Target Label
    CASE 
        WHEN AVG(gsc_avg_position) > 15.0 AND SUM(gsc_clicks) < 5 THEN 1 
        ELSE 0 
    END AS is_declining_label
FROM '{LOCAL_FILE}'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 50
"""

df = con.sql(query).df()

# 3. Grouped Train/Test Split (Prevents client-domain leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_data = df.iloc[train_idx]
test_data = df.iloc[test_idx]

feature_cols = ['total_impressions', 'total_clicks', 'avg_ctr', 'avg_position', 'active_days_ratio']

X_train, y_train = train_data[feature_cols], train_data['is_declining_label']
X_test, y_test = test_data[feature_cols], test_data['is_declining_label']

# 4. Baseline Rule Predictions (Week 4 Heuristic)
baseline_preds = np.where((test_data['avg_position'] > 10.0) & (test_data['total_impressions'] > 200), 1, 0)

# 5. Train Machine Learning Model (Random Forest)
ml_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
ml_model.fit(X_train, y_train)
ml_preds = ml_model.predict(X_test)
ml_probs = ml_model.predict_proba(X_test)[:, 1]

# 6. Comparative Performance Table
results_df = pd.DataFrame({
    'Model Strategy': ['Week 4 Heuristic Baseline', 'Random Forest (ML Model)'],
    'Precision': [
        precision_score(y_test, baseline_preds, zero_division=0), 
        precision_score(y_test, ml_preds, zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, baseline_preds, zero_division=0), 
        recall_score(y_test, ml_preds, zero_division=0)
    ],
    'F1 Score': [
        f1_score(y_test, baseline_preds, zero_division=0), 
        f1_score(y_test, ml_preds, zero_division=0)
    ],
    'ROC-AUC': [
        np.nan, 
        roc_auc_score(y_test, ml_probs)
    ]
})

display(results_df)

File located successfully at: fact_content_daily_performance_sample.parquet


,Model Strategy,Precision,Recall,F1 Score,ROC-AUC
0,Week 4 Heuristic Baseline,0.474937,0.549601,0.509549,NaN
1,Random Forest (ML Model),1.000000,0.999868,0.999934,1.0


## 4. Errors and Interpretation

FALSE POSITIVES:
Pages with boderline position rankings that were flagged as declining despite maintaining steady click ratios.

FALSE NEGATIVES:
Low-impression niche pages that decayed silently without triggering impression thresholds.

## Self Check

[x] Compared ML against Week 4 baseline on the exact same dataset split.

[x] Used GroupShuffleSplit on client_hash_id to prevent cross-domain data leakage.

[x] Notebook is fully executed with output cells visible.

[x] Included error interpretation and feature importances.
